# Chương 3 — Bag of Visual Words (Google Colab — Pure Julia)

Phiên bản dành riêng cho **Google Colab (Julia Runtime)** — tự động `git clone` repository, sử dụng trực tiếp dữ liệu `data/chapter3/digits8x8.csv` và `negFIN` tự viết của nhóm, hiển thị biểu đồ SVG ngay trên Colab.

### Hướng dẫn chạy trên Google Colab
1. Vào menu **Runtime $\rightarrow$ Change runtime type $\rightarrow$ chọn `Julia`** $\rightarrow$ bấm **Save**.
2. (Tùy chọn) Chỉnh các biến trong khối **SETUP & CONFIG** ở Phần 1 (`ALGO`, `VISUAL_MINSUP`, `VISUAL_WORDS`, `VISUAL_SEED`, `REPO_BRANCH`).
3. Bấm **Runtime $\rightarrow$ Run all** để chạy kiểm thử Chương 3, khai thác mẫu ảnh đặc trưng và hiển thị biểu đồ SVG.
4. Để chia sẻ nộp bài: **File $\rightarrow$ Save a copy in Drive** $\rightarrow$ **Share $\rightarrow$ Anyone with the link (Viewer)**.

### Mục lục các phần trong Notebook
- **Phần 1:** Thiết lập môi trường (`SETUP & CONFIG`), tự động `git clone` và nạp module `Applications`.
- **Phần 2:** Chạy bộ kiểm thử tự động Chương 3 (`tests/test_ch3_julia.jl`).
- **Phần 3:** Chạy pipeline Bag of Visual Words trên `UCI Digits 8×8`, in độ chính xác và hiển thị biểu đồ SVG.

---
## Phần 1: Thiết Lập Môi Trường & Nạp Module `Applications`

Cell này tự động xác định thư mục dự án (hoặc `git clone` từ GitHub khi mở độc lập trên Colab), kích hoạt `Project.toml` và nạp `src/applications/applications.jl`.

In [ ]:
using Pkg: Pkg

# ==============================================================================
# SETUP & CONFIG (CHỈNH THAM SỐ TẠI ĐÂY NẾU CẦN)
# ==============================================================================
const GITHUB_REPO_URL = "https://github.com/Thanhtung46/Frequent_Itemset_Mining.git"
const REPO_BRANCH = "main"
const ALGO = :optimized      # :optimized hoặc :baseline
const VISUAL_MINSUP = 0.12   # Ngưỡng support tối thiểu cho visual patterns (12%)
const VISUAL_WORDS = 16      # Số cụm từ thị giác K trong K-means
const VISUAL_SEED = 42       # Seed cố định đảm bảo tái lập

# Xác định thư mục gốc dự án (tự động git clone nếu mở độc lập trên Colab)
function resolve_project_root()::String
    dir = pwd()
    for _ in 1:4
        if isfile(joinpath(dir, "src", "applications", "applications.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && break
        dir = parent
    end
    clone_dir = joinpath(pwd(), "Frequent_Itemset_Mining")
    if !isdir(clone_dir)
        println("Đang clone mã nguồn dự án từ GitHub cho môi trường Colab...")
        run(`git clone --depth 1 --branch $REPO_BRANCH $GITHUB_REPO_URL $clone_dir`)
    end
    return clone_dir
end

const ROOT_DIR = resolve_project_root()
Pkg.activate(ROOT_DIR)
include(joinpath(ROOT_DIR, "src", "applications", "applications.jl"))
using .Applications: run_visual_words

println("Julia version: ", VERSION)
println("Project root:  ", ROOT_DIR)

---
## Phần 2: Chạy Kiểm Thử Tự Động Chương 3 (`tests/test_ch3_julia.jl`)

Kiểm tra tích hợp trực tiếp với `negFIN`, tính tái lập của bộ mô tả gradient + K-means và toàn bộ pipeline Chương 3.

In [ ]:
include(joinpath(ROOT_DIR, "tests", "test_ch3_julia.jl"))

---
## Phần 3: Chạy Bag of Visual Words & Đánh Giá Phân Loại

- Sử dụng **9 dense gradient descriptors 6 chiều/ảnh** (**không phải SIFT/ORB**), học codebook K-means thuần Julia chỉ trên tập train để tránh *data leakage*.
- Khai thác tổ hợp từ thị giác bằng `negFIN`, tính *Enrichment* theo lớp và đánh giá bộ phân loại *Nearest-Centroid* trên tập test độc lập.

In [ ]:
visual_res = run_visual_words(;
    minsup=VISUAL_MINSUP,
    words=VISUAL_WORDS,
    seed=VISUAL_SEED,
    algo=ALGO,
)

visual_dir = joinpath(ROOT_DIR, "results", "chapter3", "visual_words")
println("\n=== VISUAL WORDS SUMMARY ===")
println(read(joinpath(visual_dir, "summary.txt"), String))
println("=== SO SÁNH ĐỘ CHÍNH XÁC NEAREST-CENTROID ===")
println(read(joinpath(visual_dir, "test_accuracy_comparison.csv"), String))
println("=== TOP-10 PATTERN ĐẶC TRƯNG (ENRICHMENT) ===")
for line in Iterators.take(eachline(joinpath(visual_dir, "patterns_by_class.csv")), 11)
    println(line)
end

display("image/svg+xml", read(joinpath(visual_dir, "enriched_patterns.svg"), String))
display("image/svg+xml", read(joinpath(visual_dir, "digits_samples.svg"), String))